# 05 — Pattern Discovery & Validation**Goal:** Map the 31 feature clusters (FC1–FC31) from NB04 to failure events and subsystems, validate their relevance, and prepare a summary for expert verification with Wiener Linien.**Pipeline:** NB01 → NB02 → NB03 → **NB04** (feature clustering) → **NB05** (pattern discovery & validation)

In [ ]:
import sys, os, gc, jsonimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltimport matplotlib.dates as mdatesimport seaborn as snsROOT = os.path.abspath("..")OUT = os.path.join(ROOT, "outputs")ARTIFACT_DIR = os.path.join(OUT, "nb04_artifacts")NB05_OUT = os.path.join(OUT, "nb05_artifacts")os.makedirs(NB05_OUT, exist_ok=True)# Load NB04 artifactsall_sup = pd.read_parquet(os.path.join(ARTIFACT_DIR, "all_sup.parquet"))all_sup["window_end"] = pd.to_datetime(all_sup["window_end"])anom_activations = pd.read_parquet(os.path.join(ARTIFACT_DIR, "anom_activations.parquet"))anom_meta = pd.read_parquet(os.path.join(ARTIFACT_DIR, "anom_meta.parquet"))anom_orig_idx = np.load(os.path.join(ARTIFACT_DIR, "anom_orig_idx.npy"))with open(os.path.join(ARTIFACT_DIR, "active_feats.json")) as f:    active_feats = json.load(f)with open(os.path.join(ARTIFACT_DIR, "feature_cols.json")) as f:    feature_cols = json.load(f)with open(os.path.join(ARTIFACT_DIR, "feat_subsystem.json")) as f:    feat_subsystem = json.load(f)with open(os.path.join(ARTIFACT_DIR, "heuristic_a.json")) as f:    fc_clusters = json.load(f)with open(os.path.join(ARTIFACT_DIR, "constants.json")) as f:    constants = json.load(f)Z_THRESH = constants["Z_THRESH"]MIN_ACTIVATED = constants["MIN_ACTIVATED"]SPLIT_DATE = pd.Timestamp(constants["SPLIT_DATE"])clusters = {int(k): v for k, v in fc_clusters.items()}anom_activations.index = pd.Index(anom_orig_idx)fc_ids_sorted = sorted(clusters.keys())def get_subsystem(name):    return feat_subsystem.get(name, "Unknown")print(f"Loaded: {len(all_sup):,} windows, {len(feature_cols)} features")print(f"Anomalous windows: {len(anom_activations):,}")print(f"Feature clusters (FCs): {len(clusters)}")print(f"Features in clusters: {sum(len(v) for v in clusters.values())}")

## 1. FC Overview: Size, Subsystem Composition & Activation RatesFor each FC: how many features, which subsystems, and how often it fires.

In [ ]:
rows = []for fc_id in fc_ids_sorted:    feats = clusters[fc_id]    subs = [get_subsystem(f) for f in feats]    sub_counts = pd.Series(subs).value_counts()    dominant_sub = sub_counts.index[0]    fc_active = anom_activations[feats].any(axis=1)    act_rate = fc_active.mean()    fc_labels = all_sup.loc[anom_activations.index[fc_active], "label"]    fail_rate = fc_labels.mean() if len(fc_labels) > 0 else 0    rows.append({        "FC": f"FC{fc_id}", "n_features": len(feats),        "dominant_subsystem": dominant_sub,        "subsystems": ", ".join(f"{s}({c})" for s, c in sub_counts.items()),        "activation_rate": act_rate, "n_active_windows": fc_active.sum(),        "failure_rate": fail_rate, "example_feature": feats[0][:45]    })fc_summary = pd.DataFrame(rows)print(fc_summary[["FC", "n_features", "dominant_subsystem", "activation_rate", "failure_rate"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))fc_ids = [f"FC{i}" for i in fc_ids_sorted]sizes = [len(clusters[i]) for i in fc_ids_sorted]ax = axes[0]colors = ["#2ecc71" if s > 1 else "#95a5a6" for s in sizes]ax.barh(fc_ids, sizes, color=colors)ax.set_xlabel("Number of features")ax.set_title("FC Sizes (green=multi-feature)")ax.invert_yaxis()ax = axes[1]sub_colors = {"Brake": "#e74c3c", "Leveling": "#3498db", "APU": "#2ecc71", "Traction": "#f39c12"}sub_matrix_list = []for fc_id in fc_ids_sorted:    subs = [get_subsystem(f) for f in clusters[fc_id]]    counts = pd.Series(subs).value_counts()    sub_matrix_list.append({s: counts.get(s, 0) for s in sub_colors})sub_df = pd.DataFrame(sub_matrix_list, index=fc_ids)sub_df = sub_df[[c for c in sub_colors if c in sub_df.columns and sub_df[c].sum() > 0]]sub_df.plot.barh(stacked=True, ax=ax, color=[sub_colors[c] for c in sub_df.columns])ax.set_xlabel("Number of features")ax.set_title("Subsystem Composition")ax.invert_yaxis()ax.legend(loc="lower right", fontsize=8)ax = axes[2]fail_rates = fc_summary["failure_rate"].valuesbaseline = all_sup.loc[anom_activations.index, "label"].mean()bar_colors = ["#e74c3c" if fr > baseline * 2 else "#f39c12" if fr > baseline else "#95a5a6" for fr in fail_rates]ax.barh(fc_ids, fail_rates * 100, color=bar_colors)ax.axvline(baseline * 100, color="black", ls="--", lw=1, label=f"Baseline ({baseline*100:.1f}%)")ax.set_xlabel("Failure rate (%)")ax.set_title("Failure Rate per FC")ax.invert_yaxis()ax.legend(fontsize=8)plt.tight_layout()plt.savefig(os.path.join(NB05_OUT, "fc_overview.png"), dpi=150, bbox_inches="tight")plt.show()print("Saved fc_overview.png")

## 2. Link FCs to Failure EventsLoad raw failure event timestamps and map each anomalous window to the nearest failure event. Cross-tabulate: which FCs fire before/during failures?

In [ ]:
import globTS_COL = "TIMESTAMP"FAIL_COL = "TRAIN_IS_IN_FAILURE"FAIL_TYPE = "TRAIN_FAILURE_TYPE"HORIZON_S = 22326  # ~6.2 hours prediction horizonall_files = sorted(    glob.glob(os.path.join(ROOT, "train", "**", "*.parquet"), recursive=True) +    glob.glob(os.path.join(ROOT, "test", "**", "*.parquet"), recursive=True))chunks = []for f in all_files:    df = pd.read_parquet(f, columns=[TS_COL, FAIL_COL, FAIL_TYPE])    chunks.append(df)raw_meta = pd.concat(chunks, ignore_index=True)del chunks; gc.collect()raw_meta[TS_COL] = pd.to_datetime(raw_meta[TS_COL])raw_meta.sort_values(TS_COL, inplace=True)mask = raw_meta[FAIL_COL].astype(bool)grp = (mask != mask.shift()).cumsum()fail_events = (    raw_meta[mask].groupby(grp[mask])    .agg(start=(TS_COL, "min"), end=(TS_COL, "max"),         event_type=(FAIL_TYPE, "first"), n_rows=(TS_COL, "count"))    .reset_index(drop=True))merged = []for _, row in fail_events.iterrows():    if merged and row["event_type"] == merged[-1]["event_type"] and \       (row["start"] - merged[-1]["end"]).total_seconds() < 3600:        merged[-1]["end"] = row["end"]        merged[-1]["n_rows"] += row["n_rows"]    else:        merged.append(dict(row))fail_events = pd.DataFrame(merged)fail_events["duration_min"] = (fail_events["end"] - fail_events["start"]).dt.total_seconds() / 60fail_events["period"] = fail_events["start"].apply(lambda x: "train" if x < SPLIT_DATE else "test")fail_events.index += 1fail_events.index.name = "event_id"del raw_meta; gc.collect()print(f"Failure events: {len(fail_events)}")print(f"  Train: {(fail_events['period']=='train').sum()}, Test: {(fail_events['period']=='test').sum()}")print(f"\nEvent types:")print(fail_events["event_type"].value_counts().to_string())

In [ ]:
def find_nearest_failure(window_end, fail_events, horizon_s):    horizon_end = window_end + pd.Timedelta(seconds=horizon_s)    for idx, ev in fail_events.iterrows():        if ev["start"] >= window_end and ev["start"] <= horizon_end:            return idx, ev["event_type"], "precursor"        if ev["start"] <= window_end and ev["end"] >= window_end:            return idx, ev["event_type"], "during"    return None, None, "none"anom_meta = anom_meta.copy()anom_meta["fail_event_id"] = Noneanom_meta["fail_event_type"] = Noneanom_meta["fail_relation"] = "none"print("Mapping anomalous windows to failure events...")for i, row in anom_meta.iterrows():    eid, etype, rel = find_nearest_failure(row["window_end"], fail_events, HORIZON_S)    anom_meta.at[i, "fail_event_id"] = eid    anom_meta.at[i, "fail_event_type"] = etype    anom_meta.at[i, "fail_relation"] = relprint(f"\nAnomalous window - failure relation:")print(anom_meta["fail_relation"].value_counts().to_string())

In [ ]:
print("FC to Failure Event Mapping")print("=" * 100)fc_fail_matrix = []for fc_id in fc_ids_sorted:    feats = clusters[fc_id]    fc_active_mask = anom_activations[feats].any(axis=1)    fc_meta = anom_meta.loc[fc_active_mask]    n_total = fc_active_mask.sum()    n_precursor = (fc_meta["fail_relation"] == "precursor").sum()    n_during = (fc_meta["fail_relation"] == "during").sum()    n_none = (fc_meta["fail_relation"] == "none").sum()    covered_events = fc_meta[fc_meta["fail_relation"].isin(["precursor", "during"])]["fail_event_id"].dropna().unique()    fc_fail_matrix.append({        "FC": f"FC{fc_id}", "n_windows": n_total,        "precursor": n_precursor, "during": n_during, "none": n_none,        "fail_pct": (n_precursor + n_during) / n_total * 100 if n_total > 0 else 0,        "events_covered": len(covered_events), "event_ids": list(covered_events)    })    if n_precursor + n_during > 0:        print(f"\nFC{fc_id} ({len(feats)} feats): {n_total} windows, "              f"{n_precursor} precursor, {n_during} during, "              f"{n_none} unrelated | covers {len(covered_events)} events")fc_fail_df = pd.DataFrame(fc_fail_matrix)print(f"\n\nFCs with failure association: {(fc_fail_df['fail_pct'] > 0).sum()} / {len(fc_fail_df)}")

## 3. Subsystem MappingVisualize how FCs map to physical subsystems. Do data-driven clusters align with domain structure?

In [ ]:
subsystems = sorted(set(feat_subsystem.values()))sub_color_map = {"Brake": "#e74c3c", "Leveling": "#3498db", "APU": "#2ecc71",                 "Traction": "#f39c12", "Context": "#9b59b6"}sub_matrix = np.zeros((len(fc_ids_sorted), len(subsystems)))for i, fc_id in enumerate(fc_ids_sorted):    for feat in clusters[fc_id]:        sub = get_subsystem(feat)        if sub in subsystems:            j = subsystems.index(sub)            sub_matrix[i, j] += 1purities = []dom_subs = []for i in range(len(fc_ids_sorted)):    row = sub_matrix[i]    total = row.sum()    purity = row.max() / total if total > 0 else 0    dom_sub = subsystems[np.argmax(row)] if total > 0 else "Unknown"    purities.append(purity)    dom_subs.append(dom_sub)fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={"width_ratios": [2, 1]})ax = axes[0]sns.heatmap(sub_matrix, ax=ax, cmap="YlOrRd", annot=True, fmt=".0f",            xticklabels=subsystems, yticklabels=[f"FC{i}" for i in fc_ids_sorted],            cbar_kws={"label": "Feature count"})ax.set_title("FC x Subsystem Feature Count")ax = axes[1]colors = [sub_color_map.get(s, "#95a5a6") for s in dom_subs]ax.barh([f"FC{i}" for i in fc_ids_sorted], purities, color=colors)ax.axvline(1.0, color="green", ls="--", alpha=0.5)ax.set_xlabel("Subsystem purity (1.0 = single subsystem)")ax.set_title("FC Subsystem Purity")ax.invert_yaxis()plt.tight_layout()plt.savefig(os.path.join(NB05_OUT, "fc_subsystem_map.png"), dpi=150, bbox_inches="tight")plt.show()n_pure = sum(1 for p in purities if p == 1.0)print(f"\nPure single-subsystem FCs: {n_pure}/{len(clusters)}")print(f"Mean purity: {np.mean(purities):.2f}")

## 4. FC Validation ScorecardFor each FC: failure association, subsystem coherence, event coverage. Classification: **KEEP** vs **REVIEW**.

In [ ]:
baseline_fail = all_sup.loc[anom_activations.index, "label"].mean()print(f"Baseline failure rate among anomalous windows: {baseline_fail*100:.1f}%")scorecard = []for i, fc_id in enumerate(fc_ids_sorted):    feats = clusters[fc_id]    fc_row = fc_fail_df[fc_fail_df["FC"] == f"FC{fc_id}"].iloc[0]    fail_rate = fc_summary[fc_summary["FC"] == f"FC{fc_id}"]["failure_rate"].values[0]    purity = purities[i]    dom_sub = dom_subs[i]    events_covered = fc_row["events_covered"]    fail_score = "HIGH" if fail_rate > baseline_fail * 2 else "MED" if fail_rate > baseline_fail else "LOW"    verdict = "KEEP" if fail_score in ("HIGH", "MED") and events_covered > 0 else "REVIEW"    scorecard.append({        "FC": f"FC{fc_id}", "n_feats": len(feats), "subsystem": dom_sub,        "purity": f"{purity:.0%}", "fail_rate": f"{fail_rate*100:.1f}%",        "fail_assoc": fail_score, "events": events_covered, "verdict": verdict    })scorecard_df = pd.DataFrame(scorecard)print("\n" + scorecard_df.to_string(index=False))print(f"\nKEEP: {(scorecard_df['verdict']=='KEEP').sum()}, REVIEW: {(scorecard_df['verdict']=='REVIEW').sum()}")

## 5. Timeline: FC Activations vs Failure EventsShow when each FC fires over the full observation period, overlaid with failure events.

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))cmap = plt.cm.tab20(np.linspace(0, 1, len(fc_ids_sorted)))for row_idx, fc_id in enumerate(fc_ids_sorted):    feats = clusters[fc_id]    fc_active_mask = anom_activations[feats].any(axis=1)    fc_times = anom_meta.loc[fc_active_mask, "window_end"]    ax.scatter(fc_times, [row_idx] * len(fc_times), s=2, alpha=0.4, color=cmap[row_idx])for _, ev in fail_events.iterrows():    ax.axvspan(ev["start"], ev["end"], color="red", alpha=0.15)ax.axvline(SPLIT_DATE, color="black", ls="--", lw=1.5, label="Train/Test split")ax.set_yticks(range(len(fc_ids_sorted)))ax.set_yticklabels([f"FC{i}" for i in fc_ids_sorted], fontsize=8)ax.set_xlabel("Time")ax.set_title("FC Activation Timeline (red shading = failure events)")ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))plt.xticks(rotation=45)plt.tight_layout()plt.savefig(os.path.join(NB05_OUT, "fc_timeline.png"), dpi=150, bbox_inches="tight")plt.show()

## 6. Deep Dive: Key FCs- **FC28** (early warning): fires before failures, variance ramps up- **FC24** (failure-specific): highest failure-day activation increase (+28.7pp)- **FC19** (detection-only): fires during failures, not before

In [ ]:
def deep_dive_fc(fc_id, n_top=5):    feats = clusters[fc_id]    fc_active = anom_activations[feats].any(axis=1)    fc_meta_local = anom_meta.loc[fc_active]    freq = anom_activations.loc[fc_active, feats].mean().sort_values(ascending=False)    top_feats = freq.head(min(n_top, len(feats)))    print(f"\n{'='*80}")    print(f"FC{fc_id}: {len(feats)} features, {fc_active.sum()} active windows")    print(f"Failure relation: {fc_meta_local['fail_relation'].value_counts().to_dict()}")    print(f"Subsystems: {pd.Series([get_subsystem(f) for f in feats]).value_counts().to_dict()}")    print(f"\nTop features (activation rate):")    for feat, rate in top_feats.items():        print(f"  {feat}: {rate:.1%}")    n_plots = len(top_feats)    fig, axes = plt.subplots(n_plots, 1, figsize=(20, 3 * n_plots), sharex=True)    if n_plots == 1: axes = [axes]    for ax, feat in zip(axes, top_feats.index):        if feat in all_sup.columns:            daily = all_sup.groupby(all_sup["window_end"].dt.date)[feat].agg(["mean", "std"])            daily.index = pd.to_datetime(daily.index)            ax.plot(daily.index, daily["mean"], color="steelblue", lw=1, label="Daily mean")            ax.fill_between(daily.index, daily["mean"] - daily["std"], daily["mean"] + daily["std"],                          alpha=0.2, color="steelblue")            for _, ev in fail_events.iterrows():                ax.axvspan(ev["start"], ev["end"], color="red", alpha=0.15)            ax.set_ylabel(feat[:30], fontsize=8)    axes[0].set_title(f"FC{fc_id} - Top Feature Daily Trends (red = failure events)")    plt.tight_layout()    plt.savefig(os.path.join(NB05_OUT, f"fc{fc_id}_deep_dive.png"), dpi=150, bbox_inches="tight")    plt.show()for fc_id in [28, 24, 19]:    if fc_id in clusters:        deep_dive_fc(fc_id)

## 7. Construct Subsystem-Aware Feature GroupsGroup FCs by dominant subsystem to create subsystem-aware datasets for Phase 2 modeling (RQ2).

In [ ]:
subsystem_groups = {}for i, fc_id in enumerate(fc_ids_sorted):    sub = dom_subs[i]    if sub not in subsystem_groups:        subsystem_groups[sub] = {"fcs": [], "features": []}    subsystem_groups[sub]["fcs"].append(fc_id)    subsystem_groups[sub]["features"].extend(clusters[fc_id])print("Subsystem-Aware Feature Groups for Phase 2:")print("=" * 80)for sub, info in sorted(subsystem_groups.items()):    fc_names = [f"FC{i}" for i in info["fcs"]]    print(f"\n{sub}:")    print(f"  FCs: {fc_names}")    print(f"  Features: {len(info['features'])}")    print(f"  Sample: {info['features'][:3]}")groups_json = {sub: {"fcs": info["fcs"], "features": info["features"]}               for sub, info in subsystem_groups.items()}with open(os.path.join(NB05_OUT, "subsystem_feature_groups.json"), "w") as f:    json.dump(groups_json, f, indent=2)print(f"\nSaved subsystem_feature_groups.json")

## 8. Expert Verification SummarySummary table and key findings for the Wiener Linien meeting.

In [ ]:
print("=" * 90)print("ANOMALY PATTERN DISCOVERY - SUMMARY FOR EXPERT VERIFICATION")print("=" * 90)print(f"\nDataset: {len(all_sup):,} windows ({(all_sup['split']=='train').sum():,} train, {(all_sup['split']=='test').sum():,} test)")print(f"Anomalous windows: {len(anom_activations):,} ({len(anom_activations)/len(all_sup)*100:.1f}%)")print(f"Feature clusters: {len(clusters)} (FC1-FC{max(clusters.keys())})")print(f"Total features in clusters: {sum(len(v) for v in clusters.values())}")print(f"Failure events: {len(fail_events)}")print(f"\n--- Scorecard ---")keep = scorecard_df[scorecard_df["verdict"] == "KEEP"]print(f"KEEP: {len(keep)} FCs")for _, row in keep.iterrows():    print(f"  {row['FC']}: {row['subsystem']}, {row['fail_rate']} failure rate, {row['events']} events")print(f"\n--- Key Findings ---")print("1. FC28 (Load Signal): early warning - fires BEFORE failures with variance ramp-up")print("2. FC24 (Proportional Valve): highest failure-day activation increase (+28.7pp)")print("3. FC19 (Spring Brake): detection-only - fires DURING failures, no predictive lead time")print("4. Most failures detected in Standing state (pressure leaks visible without motion noise)")print("5. Braking clusters (FC4-FC9) may reflect normal deceleration, not degradation")print(f"\n--- Questions for Experts ---")print("Q1: Do the FC groupings align with known failure modes?")print("Q2: Are braking-related FCs (FC4-FC9) capturing real anomalies or normal operations?")print("Q3: Is the load signal (FC28) a known early indicator of pneumatic issues?")print("Q4: Which FCs should be prioritized for the predictive model?")

In [ ]:
scorecard_df.to_csv(os.path.join(NB05_OUT, "fc_scorecard.csv"), index=False)fc_fail_df.to_csv(os.path.join(NB05_OUT, "fc_failure_mapping.csv"), index=False)fc_summary.to_csv(os.path.join(NB05_OUT, "fc_summary.csv"), index=False)anom_meta.to_parquet(os.path.join(NB05_OUT, "anom_meta_enriched.parquet"))fail_events.to_parquet(os.path.join(NB05_OUT, "fail_events.parquet"))print(f"Saved NB05 artifacts to {NB05_OUT}:")for f in sorted(os.listdir(NB05_OUT)):    sz = os.path.getsize(os.path.join(NB05_OUT, f))    print(f"  {f}: {sz:,} bytes")print("\nNB05 complete. Ready for expert verification meeting.")

## Traceability: Pipeline Position**NB01 > NB02 > NB03 > NB04 > NB05**### What this notebook provides:- FC-to-failure event mapping (which FCs fire near which failures)- Subsystem alignment analysis (do data-driven clusters match physical subsystems?)- Validation scorecard (KEEP vs REVIEW for each FC)- Subsystem-aware feature groups for Phase 2 modeling- Expert verification summary and questions for Wiener Linien### Outputs:- `outputs/nb05_artifacts/fc_scorecard.csv`- `outputs/nb05_artifacts/fc_failure_mapping.csv`- `outputs/nb05_artifacts/subsystem_feature_groups.json`- `outputs/nb05_artifacts/anom_meta_enriched.parquet`- `outputs/nb05_artifacts/fail_events.parquet`- Plots: `fc_overview.png`, `fc_subsystem_map.png`, `fc_timeline.png`, `fc*_deep_dive.png`